# LAB 04 — Experiment 2: Baseline TF-IDF + Logistic Regression
Notebook cho mục 14. Nạp imdb.zip, làm split phân tầng 60/20/20, huấn luyện baseline và in thời gian train, vocabulary size, accuracy, precision, recall, F1.

ZIP hiện có chứa CSV tổng hợp (`IMDB Dataset.csv`, cột `review`, `sentiment`) chứ không còn split gốc. Vì vậy notebook loại review trùng, sau đó chia lại với `random_state=42`.

In [1]:
from pathlib import Path
import zipfile, time
import pandas as pd

ZIP_PATH = Path('/content/imdb.zip')

if not ZIP_PATH.exists():
    from google.colab import files
    print('Chọn imdb.zip để tải lên Colab.')
    uploaded = files.upload()
    ZIP_PATH = Path(next(iter(uploaded)))
print(f'Đang dùng: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1024**2:.2f} MB)')

Chọn imdb.zip để tải lên Colab.


Saving imdb.zip to imdb.zip
Đang dùng: imdb.zip (25.71 MB)


## 1. Đọc và chuẩn bị dữ liệu
Đọc các CSV trong ZIP. Hỗ trợ cột `review/sentiment` hoặc `text/label`. Gộp các file CSV tìm thấy thành một dataframe, chuẩn hóa hai nhãn và xóa review trùng trước khi chia tập.

In [2]:
def load_imdb_csv_zip(zip_path):
    frames = []
    with zipfile.ZipFile(zip_path) as z:
        csv_names = [name for name in z.namelist() if name.lower().endswith('.csv')]
        if not csv_names:
            raise ValueError('Không tìm thấy CSV trong ZIP. Notebook này cần CSV có review/sentiment hoặc text/label.')
        for name in csv_names:
            with z.open(name) as f:
                part = pd.read_csv(f)
            part.columns = [str(c).strip().lower() for c in part.columns]
            if {'review', 'sentiment'}.issubset(part.columns):
                part = part[['review', 'sentiment']].copy()
            elif {'text', 'label'}.issubset(part.columns):
                part = part.rename(columns={'text': 'review', 'label': 'sentiment'})[['review', 'sentiment']].copy()
            else:
                raise ValueError(f'{name}: cần cột review/sentiment hoặc text/label; hiện có {list(part.columns)}')
            frames.append(part)
    return pd.concat(frames, ignore_index=True)

df = load_imdb_csv_zip(ZIP_PATH)
df['review'] = df['review'].fillna('').astype(str).str.strip()
df['sentiment'] = df['sentiment'].astype(str).str.strip().str.lower()
df = df[(df['review'] != '') & df['sentiment'].isin(['positive', 'negative'])].copy()
df = df.drop_duplicates(subset='review').reset_index(drop=True)
df['label'] = df['sentiment'].map({'negative': 0, 'positive': 1})

print(f'Số review sau chuẩn hóa và loại trùng: {len(df):,}')
print('Phân bố nhãn:')
display(pd.DataFrame({'count': df['sentiment'].value_counts(), 'percent': (df['sentiment'].value_counts(normalize=True) * 100).round(2)}))

Số review sau chuẩn hóa và loại trùng: 49,582
Phân bố nhãn:


,count,percent
sentiment,,
positive,24884,50.19
negative,24698,49.81


## 2. Chia train/validation/test = 60/20/20
Tách test 20% trước; sau đó lấy 25% của 80% còn lại làm validation. `stratify` giữ phân bố nhãn gần như bằng nhau giữa các tập.

In [3]:
from sklearn.model_selection import train_test_split

X = df['review']
y = df['label']
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)

print(f'Train:      {len(X_train):,} ({len(X_train)/len(df):.1%})')
print(f'Validation: {len(X_val):,} ({len(X_val)/len(df):.1%})')
print(f'Test:       {len(X_test):,} ({len(X_test)/len(df):.1%})')
print('Train label proportions (%):')
display((y_train.map({0: 'negative', 1: 'positive'}).value_counts(normalize=True) * 100).round(2).to_frame('percent'))

Train:      29,748 (60.0%)
Validation: 9,917 (20.0%)
Test:       9,917 (20.0%)
Train label proportions (%):


,percent
label,
positive,50.19
negative,49.81


## 3. Huấn luyện baseline và đo thời gian
Vectorizer được fit trên train bên trong Pipeline để tránh data leakage. Cấu hình đồng nhất với baseline trước: TF-IDF unigram, `max_features=50000`, `min_df=2`, `sublinear_tf=True`; Logistic Regression `max_iter=1000`. Thời gian bên dưới là thời gian fit toàn bộ Pipeline trên train.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)

baseline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50000, ngram_range=(1, 1), min_df=2,
        strip_accents='unicode', sublinear_tf=True
    )),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

start = time.perf_counter()
baseline.fit(X_train, y_train)
training_seconds = time.perf_counter() - start
vocabulary_size = len(baseline.named_steps['tfidf'].vocabulary_)

print(f'Training time: {training_seconds:.2f} seconds ({training_seconds/60:.2f} minutes)')
print(f'Vocabulary size: {vocabulary_size:,} features')

Training time: 11.21 seconds (0.19 minutes)
Vocabulary size: 48,487 features


## 4. Đánh giá và bảng kết quả
Validation dùng để xem baseline trong quá trình phát triển; test dành cho kết quả cuối. Positive được xem là lớp dương để tính precision, recall và F1. Confusion matrix theo thứ tự `[[TN, FP], [FN, TP]]`.

In [5]:
def evaluate_split(name, X_part, y_part):
    pred = baseline.predict(X_part)
    result = {
        'Split': name,
        'Accuracy': accuracy_score(y_part, pred),
        'Precision (positive)': precision_score(y_part, pred, zero_division=0),
        'Recall (positive)': recall_score(y_part, pred, zero_division=0),
        'F1 (positive)': f1_score(y_part, pred, zero_division=0),
    }
    print(f'\n{name} confusion matrix [[TN, FP], [FN, TP]]:')
    print(confusion_matrix(y_part, pred, labels=[0, 1]))
    print(f'\n{name} classification report:')
    print(classification_report(y_part, pred, target_names=['negative', 'positive'], digits=4, zero_division=0))
    return result

validation_result = evaluate_split('Validation', X_val, y_val)
test_result = evaluate_split('Test', X_test, y_test)

results = pd.DataFrame([
    {'Model': 'TF-IDF + Logistic Regression', 'Training time (s)': round(training_seconds, 2),
     'Vocabulary size': vocabulary_size, 'Split': row['Split'],
     'Accuracy': row['Accuracy'], 'Precision': row['Precision (positive)'],
     'Recall': row['Recall (positive)'], 'F1': row['F1 (positive)']}
    for row in [validation_result, test_result]
])
display(results.style.format({
    'Training time (s)': '{:.2f}', 'Accuracy': '{:.4f}',
    'Precision': '{:.4f}', 'Recall': '{:.4f}', 'F1': '{:.4f}'
}))


Validation confusion matrix [[TN, FP], [FN, TP]]:
[[4354  586]
 [ 469 4508]]

Validation classification report:
              precision    recall  f1-score   support

    negative     0.9028    0.8814    0.8919      4940
    positive     0.8850    0.9058    0.8952      4977

    accuracy                         0.8936      9917
   macro avg     0.8939    0.8936    0.8936      9917
weighted avg     0.8938    0.8936    0.8936      9917


Test confusion matrix [[TN, FP], [FN, TP]]:
[[4362  578]
 [ 447 4530]]

Test classification report:
              precision    recall  f1-score   support

    negative     0.9070    0.8830    0.8949      4940
    positive     0.8868    0.9102    0.8984      4977

    accuracy                         0.8966      9917
   macro avg     0.8969    0.8966    0.8966      9917
weighted avg     0.8969    0.8966    0.8966      9917



,Model,Training time (s),Vocabulary size,Split,Accuracy,Precision,Recall,F1
0,TF-IDF + Logistic Regression,11.21,48487,Validation,0.8936,0.8850,0.9058,0.8952
1,TF-IDF + Logistic Regression,11.21,48487,Test,0.8966,0.8868,0.9102,0.8984


## Trả lời mục 14
Baseline là TF-IDF + Logistic Regression. Sau khi chạy notebook, bảng trên ghi thời gian huấn luyện, kích thước vocabulary và các chỉ số trên validation/test. Khi báo cáo kết quả cuối, dùng hàng Test; không chọn mô hình chỉ dựa trên test vì test cần giữ độc lập cho đánh giá cuối cùng. Confusion matrix và F1/precision/recall được báo cáo cùng accuracy để mô tả hiệu năng đầy đủ.